In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# 1. Paths
# ============================================================

INPUT_PATH = Path("../results/one_class_neutral_model_results/one_class_neutral_scores_v1.tsv")

OUTPUT_DIR = Path("../results/mutational_spectrum_groups")
OUTPUT_DIR.mkdir(exist_ok=True)

GROUPED_OUTPUT = OUTPUT_DIR / "spectrum_groups_from_one_class_T95_v1.tsv"
GROUP_COUNTS_OUTPUT = OUTPUT_DIR / "spectrum_group_counts_v1.tsv"

# ============================================================
# 2. Load one-class results
# ============================================================

df = pd.read_csv(INPUT_PATH, sep="\t", low_memory=False)

print("Input shape:", df.shape)

print("\nAnalysis group distribution:")
print(df["analysis_group"].value_counts(dropna=False))

# ============================================================
# 3. Required columns
# ============================================================

required_cols = [
    "variant_id",
    "position",
    "reference",
    "alternate",
    "validation_label",
    "is_neutral_dataset8",
    "is_pathogenic_dataset9",
    "is_disease_suspected_dataset3",
    "analysis_group",

    "isolation_forest_outlier_score",
    "isolation_forest_above_T90",
    "isolation_forest_above_T95",
    "isolation_forest_above_T99",

    "mlc_score",
    "pop_af_max",
    "pop_af_hom_max",
    "pop_af_het_max",
    "rarity_soft",
    "no_homoplasmic_signal",
]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

work = df.copy()

Input shape: (49704, 30)

Analysis group distribution:
analysis_group
unlabeled_or_other            41280
neutral_reference              8284
article_pathogenic_posthoc       84
disease_suspected_posthoc        56
Name: count, dtype: int64


In [2]:
# ============================================================
# 4. Basic masks
# ============================================================

is_neutral_reference = work["analysis_group"] == "neutral_reference"
is_unlabeled = work["analysis_group"] == "unlabeled_or_other"
is_article_pathogenic = work["analysis_group"] == "article_pathogenic_posthoc"
is_disease_suspected = work["analysis_group"] == "disease_suspected_posthoc"

below_T95 = work["isolation_forest_above_T95"] == 0
above_T95 = work["isolation_forest_above_T95"] == 1

below_T99 = work["isolation_forest_above_T99"] == 0
above_T99 = work["isolation_forest_above_T99"] == 1

In [3]:
# ============================================================
# 5. Define spectrum analysis groups
# ============================================================

work["spectrum_group_primary_T95"] = "exclude"

# ------------------------------------------------------------
# Primary neutral-like group:
# all known neutral_reference + unlabeled below T95
# ------------------------------------------------------------

work.loc[
    is_neutral_reference | (is_unlabeled & below_T95),
    "spectrum_group_primary_T95"
] = "expanded_neutral_like_T95"

# ------------------------------------------------------------
# Primary out-of-neutral-domain group:
# only unlabeled above T95
# ------------------------------------------------------------

work.loc[
    is_unlabeled & above_T95,
    "spectrum_group_primary_T95"
] = "unlabeled_out_of_neutral_domain_T95"

# article_pathogenic and disease_suspected remain excluded
# from primary binary comparison

In [4]:
# ============================================================
# 6. Strict sensitivity group
# ============================================================

work["spectrum_group_strict_T95"] = "exclude"

work.loc[
    (is_neutral_reference & below_T95) | (is_unlabeled & below_T95),
    "spectrum_group_strict_T95"
] = "strict_expanded_neutral_like_T95"

work.loc[
    is_unlabeled & above_T95,
    "spectrum_group_strict_T95"
] = "unlabeled_out_of_neutral_domain_T95"

In [5]:
# ============================================================
# 7. T99 sensitivity group
# ============================================================

work["spectrum_group_primary_T99"] = "exclude"

work.loc[
    is_neutral_reference | (is_unlabeled & below_T99),
    "spectrum_group_primary_T99"
] = "expanded_neutral_like_T99"

work.loc[
    is_unlabeled & above_T99,
    "spectrum_group_primary_T99"
] = "unlabeled_out_of_neutral_domain_T99"

In [6]:
# ============================================================
# 8. Post-hoc biological groups
# ============================================================

work["spectrum_group_posthoc"] = "exclude"

work.loc[
    is_neutral_reference,
    "spectrum_group_posthoc"
] = "known_neutral_reference"

work.loc[
    is_article_pathogenic,
    "spectrum_group_posthoc"
] = "article_pathogenic_posthoc"

work.loc[
    is_disease_suspected,
    "spectrum_group_posthoc"
] = "disease_suspected_posthoc"

work.loc[
    is_unlabeled & above_T95,
    "spectrum_group_posthoc"
] = "unlabeled_out_of_neutral_domain_T95"

work.loc[
    is_unlabeled & below_T95,
    "spectrum_group_posthoc"
] = "unlabeled_neutral_like_T95"

In [7]:
# ============================================================
# 9. Substitution type
# ============================================================

work["reference"] = work["reference"].astype(str).str.upper()
work["alternate"] = work["alternate"].astype(str).str.upper()

valid_bases = {"A", "C", "G", "T"}

invalid_ref = ~work["reference"].isin(valid_bases)
invalid_alt = ~work["alternate"].isin(valid_bases)

if invalid_ref.any() or invalid_alt.any():
    print("\nWarning: invalid bases detected.")
    print("Invalid reference:", invalid_ref.sum())
    print("Invalid alternate:", invalid_alt.sum())

work["substitution_type_12"] = work["reference"] + ">" + work["alternate"]

substitution_order_12 = [
    "A>C", "A>G", "A>T",
    "C>A", "C>G", "C>T",
    "G>A", "G>C", "G>T",
    "T>A", "T>C", "T>G",
]

work["is_valid_snv_substitution"] = work["substitution_type_12"].isin(
    substitution_order_12
).astype(int)

print("\nInvalid substitution types:")
print(work.loc[work["is_valid_snv_substitution"] == 0, "substitution_type_12"].value_counts())


Invalid substitution types:
Series([], Name: count, dtype: int64)


In [8]:
# ============================================================
# 10. Save grouped table
# ============================================================

output_cols = [
    "variant_id",
    "position",
    "reference",
    "alternate",
    "substitution_type_12",
    "is_valid_snv_substitution",

    "validation_label",
    "is_neutral_dataset8",
    "is_pathogenic_dataset9",
    "is_disease_suspected_dataset3",
    "analysis_group",

    "isolation_forest_outlier_score",
    "isolation_forest_above_T90",
    "isolation_forest_above_T95",
    "isolation_forest_above_T99",

    "mlc_score",
    "pop_af_max",
    "pop_af_hom_max",
    "pop_af_het_max",
    "rarity_soft",
    "no_homoplasmic_signal",

    "spectrum_group_primary_T95",
    "spectrum_group_strict_T95",
    "spectrum_group_primary_T99",
    "spectrum_group_posthoc",
]

work[output_cols].to_csv(GROUPED_OUTPUT, sep="\t", index=False)

print("\nSaved grouped table:")
print(GROUPED_OUTPUT)


Saved grouped table:
../results/mutational_spectrum_groups/spectrum_groups_from_one_class_T95_v1.tsv


In [9]:
# ============================================================
# 11. Group counts
# ============================================================

count_rows = []

group_cols = [
    "analysis_group",
    "spectrum_group_primary_T95",
    "spectrum_group_strict_T95",
    "spectrum_group_primary_T99",
    "spectrum_group_posthoc",
]

for group_col in group_cols:
    counts = work[group_col].value_counts(dropna=False)

    for group_name, n in counts.items():
        count_rows.append({
            "group_column": group_col,
            "group_name": group_name,
            "n_variants": int(n),
            "fraction": float(n / work.shape[0]),
        })

group_counts = pd.DataFrame(count_rows)

group_counts.to_csv(GROUP_COUNTS_OUTPUT, sep="\t", index=False)

print("\nGroup counts:")
print(group_counts)

print("\nSaved group counts:")
print(GROUP_COUNTS_OUTPUT)


Group counts:
                  group_column                           group_name  \
0               analysis_group                   unlabeled_or_other   
1               analysis_group                    neutral_reference   
2               analysis_group           article_pathogenic_posthoc   
3               analysis_group            disease_suspected_posthoc   
4   spectrum_group_primary_T95            expanded_neutral_like_T95   
5   spectrum_group_primary_T95  unlabeled_out_of_neutral_domain_T95   
6   spectrum_group_primary_T95                              exclude   
7    spectrum_group_strict_T95     strict_expanded_neutral_like_T95   
8    spectrum_group_strict_T95  unlabeled_out_of_neutral_domain_T95   
9    spectrum_group_strict_T95                              exclude   
10  spectrum_group_primary_T99            expanded_neutral_like_T99   
11  spectrum_group_primary_T99  unlabeled_out_of_neutral_domain_T99   
12  spectrum_group_primary_T99                              ex

In [10]:
group_counts[group_counts["group_column"] == "spectrum_group_primary_T95"]

,group_column,group_name,n_variants,fraction
4,spectrum_group_primary_T95,expanded_neutral_like_T95,41264,0.830195
5,spectrum_group_primary_T95,unlabeled_out_of_neutral_domain_T95,8300,0.166989
6,spectrum_group_primary_T95,exclude,140,0.002817
